# Warm-up 2. Training on images with the GPU

The knee lessons work on MRI images. Before that, train a small model on simple
images so the parts are familiar: an image becomes a tensor, a network reads the
tensor, a training loop adjusts the weights, and the GPU does the arithmetic.

This notebook uses the **digits** dataset that ships with scikit-learn: 1797
handwritten digits, each an 8x8 grayscale image, labeled 0 to 9. It needs no
download, so it runs anywhere.

**Where you are: Warm-up 2 of 2.** After this you start Lesson 0 on the knee data.

Turn on the GPU first. In the Kaggle settings panel, set the accelerator to a GPU
(for example T4). The cell below prints which device it found. On CPU the notebook
still runs; it is only slower.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

SEED = 0
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Step 1: Load the images and look at them

Load the digits, scale the pixels to the 0 to 1 range, and show a few. Looking at
the data first is the same habit you used in Warm-up 1.

In [ ]:
from sklearn.datasets import load_digits

data = load_digits()
images = (data.images / 16.0).astype("float32")   # [N, 8, 8], pixels in 0..1
labels = data.target.astype("int64")
print("images:", images.shape, "labels:", labels.shape, "classes:", sorted(set(labels)))

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 8, figsize=(8, 1.5))
    for ax, img, lab in zip(axes, images, labels):
        ax.imshow(img, cmap="gray"); ax.set_title(int(lab)); ax.axis("off")
    plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

## Step 2: Split into train and test

Hold out a quarter of the images for testing. You never judge a model on the images
it trained on. This is the same rule as cross-validation, in its simplest form.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, random_state=SEED, stratify=labels)

def to_tensor(a, dtype):
    return torch.tensor(a, dtype=dtype, device=device)

Xtr = to_tensor(X_train.reshape(len(X_train), -1), torch.float32)   # [N, 64]
Xte = to_tensor(X_test.reshape(len(X_test), -1), torch.float32)
ytr = to_tensor(y_train, torch.long)
yte = to_tensor(y_test, torch.long)
print("train:", tuple(Xtr.shape), "test:", tuple(Xte.shape))

## Step 3: Build and train a small network

The model is two linear layers with a ReLU between them. It reads the 64 pixels and
produces a score for each of the 10 digits. The training loop is the same four steps
you will see everywhere: forward, loss, backward, step.

In [ ]:
model = nn.Sequential(
    nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)
).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(60):
    opt.zero_grad()
    loss = loss_fn(model(Xtr), ytr)
    loss.backward()
    opt.step()
    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch+1:3d}  training loss {loss.item():.3f}")

## Step 4: Test accuracy

Predict the held-out images and measure accuracy, the fraction of digits the model
labels correctly. A model that guesses would score about 0.10, since there are ten
classes.

In [ ]:
with torch.inference_mode():
    pred = model(Xte).argmax(dim=1)
acc = (pred == yte).float().mean().item()
print(f"test accuracy: {acc:.3f}  (random guess is about 0.10)")

## Experiment: how much does training length matter?

Re-train from scratch for a few epoch counts and read the test accuracy each time.
Watch the accuracy rise and then level off. This is the same knob you tune on any
model. Change the list and run it again.

In [ ]:
def train_and_score(epochs):
    torch.manual_seed(SEED)
    m = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)).to(device)
    o = torch.optim.Adam(m.parameters(), lr=0.01)
    for _ in range(epochs):
        o.zero_grad(); loss_fn(m(Xtr), ytr).backward(); o.step()
    with torch.inference_mode():
        return (m(Xte).argmax(dim=1) == yte).float().mean().item()

for e in [1, 5, 20, 60]:
    print(f"epochs={e:3d}  test accuracy {train_and_score(e):.3f}")

## Recap

You loaded images, made tensors, trained a small network on the GPU, and measured
test accuracy. In the knee lessons you keep the head and the training idea, but you
replace this tiny network with a frozen pretrained encoder that turns an MRI slice
into a feature vector. Same shape of problem: images in, a number out. Continue to
Lesson 0.